# S11 — BI/ML Distribuido | Predicciones aeronáuticas Perú

**Unidad II — Integrantes: Leonardo (tráfico en 5 minutos) y Jhonatan (velocidad siguiente).**

**Parte A:** Spark batch sobre el Parquet ADSB.lol Perú, resultados de evaluación de S10 y KPIs publicados una vez en **Pushgateway**.

**Parte B:** Streaming secundario desde los Parquet de predicciones que producen los notebooks S10; publica métricas por microlote en **Pushgateway → Prometheus → Grafana**. No reentrena ni reemplaza los modelos S10. Se verifica que el modelo de Leonardo se pueda recargar; Jhonatan ganó con persistencia (no tiene un `PipelineModel` operativo).

**Trazabilidad:** datos originales de vuelos entre el 26–30/12/2024; fechas de replay en 2026 son operativas. `opensky_live` identifica la inferencia sobre nuevos eventos. Una predicción de futuro no es una observación real del mismo instante.

**Antes de empezar:** Ejecuta las instrucciones `README_S11.md` para instalar Pushgateway y configurar el scrape en Prometheus. Mantén abiertos los notebooks S10 con los streams activos si deseas ver nuevas inferencias.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import urllib.request, json, math, time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from pyspark.ml import PipelineModel

ROOT=Path('/workspace/artifacts/s10')
S11=Path('/workspace/artifacts/s11')
S11.mkdir(parents=True, exist_ok=True)
PUSH='http://aire-pushgateway:9091'
SPARK_PACKAGE='org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0'
spark=(SparkSession.builder.appName('S11-BI-ML-Peru').master('local[2]')
    .config('spark.sql.session.timeZone','UTC')
    .config('spark.sql.shuffle.partitions','4')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

def push_group(job, metrics):
    """Reemplaza un grupo completo de gauges: nombres fijos, sin etiquetas dinámicas."""
    lines=[]
    for name,val in metrics.items():
        if val is None: continue
        v=float(val)
        if not math.isfinite(v): continue
        lines += [f'# TYPE {name} gauge', f'{name} {v}']
    if not lines: return
    body=('\n'.join(lines)+'\n').encode('utf-8')
    req=urllib.request.Request(PUSH+f'/metrics/job/{job}/instance/spark',data=body,method='PUT',headers={'Content-Type':'text/plain; version=0.0.4; charset=utf-8'})
    with urllib.request.urlopen(req,timeout=15) as res:
        assert res.status in (200,202),res.status

print('Spark',spark.version,'Pushgateway',PUSH)
print('Parquet histórico existe:', (ROOT/'peru_historico_unificado').exists())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 18:35:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/06 18:35:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.2.0 Pushgateway http://aire-pushgateway:9091
Parquet histórico existe: True


## A1. KPIs batch del dataset histórico real de Perú

Indicadores de negocio/operación: observaciones retenidas (tras deduplicación), aeronaves distintas, días cubiertos, velocidad promedio y razón de retención tras deduplicación. La razón **no** es «ratio de eventos válidos»: no existe un registro completo de descartes de origen que permita calcularlo exactamente.

In [2]:
HIST=str(ROOT/'peru_historico_unificado')
assert Path(HIST).exists(),HIST
df=spark.read.parquet(HIST)
res=df.agg(F.count('*').alias('filas'),F.countDistinct('icao24').alias('aeronaves'),
           F.countDistinct(F.to_date('eventTimestamp')).alias('dias'),
           F.avg('velocity').alias('velocidad_media'),
           F.avg('baroAltitude').alias('altitud_media')).first().asDict()
# Total antes de deduplicación observado en la ingestión de los cinco CSV:
ORIGINALES_CSV=184766
kpi={
 's11_peru_batch_observaciones':res['filas'],
 's11_peru_batch_aeronaves_distintas':res['aeronaves'],
 's11_peru_batch_dias':res['dias'],
 's11_peru_batch_retencion_dedup_ratio':res['filas']/ORIGINALES_CSV,
 's11_peru_batch_velocidad_media_mps':res['velocidad_media'],
 's11_peru_batch_altitud_media_m':res['altitud_media'],
 's11_peru_batch_actualizado_epoch':datetime.now(timezone.utc).timestamp(),
}
push_group('s11_peru_batch',kpi)
print('KPIs batch enviados al Pushgateway:')
print(json.dumps(kpi,indent=2,ensure_ascii=False))

[Stage 1:>                                                          (0 + 2) / 2]

KPIs batch enviados al Pushgateway:
{
  "s11_peru_batch_observaciones": 184753,
  "s11_peru_batch_aeronaves_distintas": 401,
  "s11_peru_batch_dias": 5,
  "s11_peru_batch_retencion_dedup_ratio": 0.9999296407347672,
  "s11_peru_batch_velocidad_media_mps": 228.85047304372267,
  "s11_peru_batch_altitud_media_m": 9974.1575740458,
  "s11_peru_batch_actualizado_epoch": 1793990134.480335
}


## A2. Comparación de modelos evaluados en S10

Leemos los CSV **generados realmente por los notebooks**. El ganador se define por el RMSE de validación (29/12), **no** por las métricas del día de prueba (30/12). Las series de Prometheus usarán el instante de publicación, mientras que estas métricas describen la evaluación histórica de 2024.

In [3]:
import csv

def cargar_tabla(nombre):
    ruta=ROOT/f'comparativa_{nombre}.csv'
    assert ruta.exists(), f'Falta ejecutar sección 7 de S10: {ruta}'
    with ruta.open(encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))

for who,slug in [('leonardo','leonardo_trafico'),('jhonatan','jhonatan_velocidad')]:
    filas=cargar_tabla(slug)
    orden={'Persistencia':'persistencia','Regresion lineal':'lineal','Random Forest':'random_forest'}
    for fila in filas:
        model=orden.get(fila['modelo'])
        if model is None: continue
        values={}
        for col in ('rmse_val','mae_val','r2_val','rmse_test','mae_test','r2_test'):
            values[f's11_{who}_{model}_{col}']=float(fila[col])
        push_group(f's11_{who}_{model}',values)
        print(who,model,values)
    sel=json.loads((ROOT/f'seleccion_{slug}.json').read_text(encoding='utf-8'))
    print('Ganador por validación:',who,sel['modelo_seleccionado'])

leonardo lineal {'s11_leonardo_lineal_rmse_val': 1.3022936937782221, 's11_leonardo_lineal_mae_val': 1.0287905860013737, 's11_leonardo_lineal_r2_val': 0.7482268519121529, 's11_leonardo_lineal_rmse_test': 1.383974992439501, 's11_leonardo_lineal_mae_test': 1.0863354131952057, 's11_leonardo_lineal_r2_test': 0.6961666724628555}
leonardo random_forest {'s11_leonardo_random_forest_rmse_val': 1.3064118922695536, 's11_leonardo_random_forest_mae_val': 1.0368985934952977, 's11_leonardo_random_forest_r2_val': 0.7466319870918146, 's11_leonardo_random_forest_rmse_test': 1.34142074433315, 's11_leonardo_random_forest_mae_test': 1.0433069161481314, 's11_leonardo_random_forest_r2_test': 0.7145638592609549}
leonardo persistencia {'s11_leonardo_persistencia_rmse_val': 1.3294213444619847, 's11_leonardo_persistencia_mae_val': 1.0034722222222223, 's11_leonardo_persistencia_r2_val': 0.7376283963853026, 's11_leonardo_persistencia_rmse_test': 1.4055638569974547, 's11_leonardo_persistencia_mae_test': 1.034843205

## A3. Verificar artefactos guardados SIN reentrenar

La inferencia en S10 ya utiliza el modelo guardado de Leonardo. Comprobamos que se puede recargar. En Jhonatan ganó persistencia: el predictor operativo es explícitamente `prediction = velocity`; no debe publicarse como ganador el modelo de regresión lineal guardado para comparación.

In [4]:
LEONARDO_MODEL=str(ROOT/'models'/'leonardo_trafico')
modelo_leonardo=PipelineModel.load(LEONARDO_MODEL)
sel_l=json.loads((ROOT/'seleccion_leonardo_trafico.json').read_text(encoding='utf-8'))
sel_j=json.loads((ROOT/'seleccion_jhonatan_velocidad.json').read_text(encoding='utf-8'))
assert sel_l['modelo_seleccionado']=='Regresion lineal'
assert sel_j['modelo_seleccionado']=='Persistencia'
print('Leonardo: modelo recargado:', LEONARDO_MODEL)
print('Jhonatan: predictor de persistencia, sin reentrenamiento')

Leonardo: modelo recargado: /workspace/artifacts/s10/models/leonardo_trafico
Jhonatan: predictor de persistencia, sin reentrenamiento


## B1. Streaming de salidas S10 → métricas en Pushgateway

**No se leen los eventos Kafka de nuevo:** S10 ya ejecuta la inferencia en Spark y guarda las predicciones en Parquet. Esta etapa realiza un **segundo streaming** sobre esas salidas, publica resultados por microlote y permite al docente comprobar la integración. Se usan checkpoints propios de S11.

**Importante:** Leonardo publica conteo **observado en ventana anterior** y pronóstico para la **siguiente** ventana. No son un par `real/predicción` de la misma ventana. Su watermark de diez minutos puede causar predicciones tardías: este dashboard mide recepción y no afirma oportunidad del pronóstico.

Jhonatan publica lectura actual y pronóstico de siguiente lectura. Para medir un error en vivo se debe recibir una **segunda observación del mismo ICAO24 en 1–90 segundos**. La frecuencia usual del productor OpenSky (~120s) puede dejar la métrica sin datos válidos.

In [5]:
# Campos reales generados por los notebooks S10
ESQ_L=(StructType()
 .add('inicio',TimestampType()).add('fin',TimestampType())
 .add('aeronaves',LongType()).add('prediction',DoubleType())
 .add('objetivo_inicio',TimestampType()).add('modelo',StringType())
 .add('fuente',StringType()).add('batch_id',LongType()))
ESQ_J=(StructType()
 .add('icao24',StringType()).add('eventTimestamp',TimestampType())
 .add('velocity',DoubleType()).add('prediction',DoubleType())
 .add('modelo',StringType()).add('fuente',StringType())
 .add('batch_id',LongType()))

RUTA_L=str(ROOT/'predicciones_live'/'leonardo')
RUTA_J=str(ROOT/'predicciones_live'/'jhonatan')
print('Leonardo disponible:',Path(RUTA_L).exists())
print('Jhonatan disponible:',Path(RUTA_J).exists())

Leonardo disponible: True
Jhonatan disponible: True


### B2. Iniciar publicadores de métricas por microlote

Ejecuta solo **una vez** esta celda, después de que existan las carpetas de predicciones S10. Si Leonardo todavía no ha generado archivos, se iniciará automáticamente cuando crees la carpeta desde S10 y vuelvas a ejecutar esta celda (sin duplicar streams). El streaming de Spark sobre Parquet funciona cuando aparecen **nuevos archivos**; los archivos anteriores se leen inicialmente gracias al checkpoint nuevo.

Las métricas se etiquetan en su nombre como `observado_actual`, `prediccion_siguiente` y `error_validado`; no se inventan errores cero.

In [6]:
# Para reiniciar el notebook, detener streams antiguos desde su propia sesión.
queries_s11 = globals().get('queries_s11', {})

def ultimo(df, order_col, cols):
    # Devuelve una fila del microlote para KPIs de bajo cardinalidad.
    xs=(df.filter(F.col(order_col).isNotNull())
          .orderBy(F.col(order_col).desc()).select(*cols).limit(1).collect())
    return xs[0].asDict() if xs else None

def publica_leonardo(lote,bid):
    n=lote.count()
    if n==0: return
    r=ultimo(lote,'inicio',['inicio','fin','aeronaves','prediction'])
    if not r: return
    push_group('s11_peru_live_leonardo',{
      's11_leonardo_stream_ventanas_lote': n,
      's11_leonardo_stream_observado_actual': r['aeronaves'],
      's11_leonardo_stream_prediccion_siguiente': r['prediction'],
      's11_leonardo_stream_ventana_fin_epoch':r['fin'].replace(tzinfo=timezone.utc).timestamp(),
      's11_leonardo_stream_ultima_publicacion_epoch':datetime.now(timezone.utc).timestamp(),
      's11_leonardo_stream_batch_id':bid,
    })
    print('[S11 Leonardo]',bid,'ventanas:',n,'observadas:',r['aeronaves'],'previstas:',r['prediction'])

def publica_jhonatan(lote,bid):
    n=lote.count()
    if n==0: return
    r=ultimo(lote,'eventTimestamp',['eventTimestamp','icao24','velocity','prediction'])
    if not r: return
    push_group('s11_peru_live_jhonatan',{
      's11_jhonatan_stream_filas_lote':n,
      's11_jhonatan_stream_velocidad_actual_mps':r['velocity'],
      's11_jhonatan_stream_prediccion_siguiente_mps':r['prediction'],
      's11_jhonatan_stream_evento_epoch':r['eventTimestamp'].replace(tzinfo=timezone.utc).timestamp(),
      's11_jhonatan_stream_ultima_publicacion_epoch':datetime.now(timezone.utc).timestamp(),
      's11_jhonatan_stream_batch_id':bid,
    })
    print('[S11 Jhonatan]',bid,'filas:',n,'observada:',r['velocity'],'prevista:',r['prediction'])

INICIAR_S11=True
if INICIAR_S11:
    if 'jhonatan' not in queries_s11 or not queries_s11['jhonatan'].isActive:
        if Path(RUTA_J).exists() and list(Path(RUTA_J).rglob('*.parquet')):
            queries_s11['jhonatan']=(spark.readStream.schema(spark.read.parquet(RUTA_J).schema).parquet(RUTA_J)
                 .writeStream.foreachBatch(publica_jhonatan).outputMode('append')
                 .option('checkpointLocation','/workspace/artifacts/checkpoints/s11_dashboard_jhonatan_v1')
                 .trigger(processingTime='30 seconds').start())
            print('Publicador Jhonatan S11 activo')
        else: print('Aún no existen predicciones Parquet S10 Jhonatan')
    if 'leonardo' not in queries_s11 or not queries_s11['leonardo'].isActive:
        if Path(RUTA_L).exists() and list(Path(RUTA_L).rglob('*.parquet')):
            queries_s11['leonardo']=(spark.readStream.schema(spark.read.parquet(RUTA_L).schema).parquet(RUTA_L)
                 .writeStream.foreachBatch(publica_leonardo).outputMode('append')
                 .option('checkpointLocation','/workspace/artifacts/checkpoints/s11_dashboard_leonardo_v1')
                 .trigger(processingTime='30 seconds').start())
            print('Publicador Leonardo S11 activo')
        else: print('Leonardo aún sin Parquet. No se inicia su stream; revisar watermark de S10.')
else:
    print('S11 solo batch; streaming no iniciado')

Publicador Jhonatan S11 activo
Publicador Leonardo S11 activo
[S11 Jhonatan] 0 filas: 8 observada: 227.86 prevista: 227.86
[S11 Leonardo] 0 ventanas: 4 observadas: 1 previstas: 0.7265092899405197


## B3. Diagnóstico y evidencias de streaming

Un stream activo **no demuestra** que ya haya predicciones. Revisa `lastProgress` (filas, batch y error) y la carpeta de resultados. Si el producer OpenSky no envía datos, el panel seguirá sin valores nuevos.

In [7]:
for nombre,q in queries_s11.items():
    print('-----',nombre,'-----')
    print('activo:',q.isActive,'error:',q.exception())
    p=q.lastProgress
    print('batch:',p.get('batchId') if p else None,
          'filas:',p.get('numInputRows') if p else None,
          'duracion_ms:',p.get('batchDuration') if p else None)
print('Para detener solo el publicador S11:')
print('for q in queries_s11.values(): q.stop()')

----- jhonatan -----
activo: True error: None
batch: 0 filas: 16 duracion_ms: 3485
----- leonardo -----
activo: True error: None
batch: 0 filas: 8 duracion_ms: 3345
Para detener solo el publicador S11:
for q in queries_s11.values(): q.stop()


## C. Verificaciones en Prometheus y Grafana

Verifica en `http://localhost:49090/targets` que `pushgateway-s11` esté UP. Consulta en Prometheus: `s11_peru_batch_observaciones`, `s11_leonardo_lineal_rmse_test`, `s11_jhonatan_persistencia_mae_test`, `s11_jhonatan_stream_prediccion_siguiente_mps`.

Importa el archivo `S11_Grafana_Peru.json` desde **Grafana → Dashboards → New → Import** y selecciona el datasource Prometheus. El dashboard identifica valores de validación y prueba históricos por separado y no presenta los valores actuales como verificaciones de predicción futura.

### Evidencias para S12

Captura: (1) KPI batch, (2) selección de modelos, (3) métricas Pushgateway, (4) tres targets Prometheus + Pushgateway, (5) evolución en vivo en Grafana, (6) `lastProgress` de los streams, (7) error en vivo solo cuando se haya podido comparar con verdad posterior. Los KPIs batch no se recalculan en cada microlote.

**Limitaciones pendientes:** la guía S11 solicita error medido en producción. En OpenSky, si el muestreo excede el horizonte 1–90 s de Jhonatan, no se puede atribuirle el RMSE histórico a las lecturas en vivo. Para Leonardo, el watermark de S10 emite pronósticos tardíos; antes de afirmar una predicción oportuna debe modificarse el pipeline S10.

In [8]:
for nombre, q in queries_s11.items():
    p = q.lastProgress
    print(
        nombre,
        "activo:", q.isActive,
        "error:", q.exception(),
        "batch:", p.get("batchId") if p else None,
        "filas:", p.get("numInputRows") if p else None
    )

jhonatan activo: True error: None batch: 1 filas: 0
leonardo activo: True error: None batch: 1 filas: 0
